In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy.optimize import minimize
%matplotlib inline


In [6]:
data = yf.download(tickers, start='2018-01-01', end='2023-01-01', auto_adjust=False)['Adj Close'].dropna()




[*********************100%***********************]  5 of 5 completed


In [7]:
returns = data.pct_change().dropna()
returns.tail()


Ticker,AAPL,AMZN,GOOGL,META,MSFT
Date,,,,,
2022-12-23,-0.002798,0.017425,0.016750,0.007855,0.002267
2022-12-27,-0.013878,-0.025924,-0.020621,-0.009827,-0.007414
2022-12-28,-0.030685,-0.014692,-0.015677,-0.010780,-0.010255
2022-12-29,0.028324,0.028844,0.028249,0.040132,0.027630
2022-12-30,0.002469,-0.002138,-0.002487,0.000665,-0.004938


In [9]:
mean_returns = returns.mean()
cov_matrix = returns.cov()


In [10]:
def portfolio_performance(weights, mean_returns, cov_matrix):
    ret = np.sum(mean_returns * weights) * 252  # Annualized return
    vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights))) * np.sqrt(252)  # Annualized volatility
    sharpe = ret / vol
    return ret, vol, sharpe

def neg_sharpe_ratio(weights, mean_returns, cov_matrix):
    return -portfolio_performance(weights, mean_returns, cov_matrix)[2]  # We minimize negative Sharpe


In [11]:
num_assets = len(tickers)
args = (mean_returns, cov_matrix)
bounds = tuple((0, 1) for _ in range(num_assets))  # Weights between 0 and 1
constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})  # Weights must sum to 1


In [12]:
initial_guess = num_assets * [1. / num_assets]  # Equal allocation
result = minimize(neg_sharpe_ratio, initial_guess, args=args,
                  method='SLSQP', bounds=bounds, constraints=constraints)

optimal_weights = result.x


In [13]:
ret, vol, sharpe = portfolio_performance(optimal_weights, mean_returns, cov_matrix)

print("📊 Optimal Portfolio Allocation:")
for ticker, weight in zip(tickers, optimal_weights):
    print(f"{ticker}: {weight:.2%}")

print(f"\n✅ Expected Annual Return: {ret:.2%}")
print(f"📉 Annual Volatility: {vol:.2%}")
print(f"🌟 Sharpe Ratio: {sharpe:.2f}")


📊 Optimal Portfolio Allocation:
AAPL: 48.49%
MSFT: 0.00%
GOOGL: 0.00%
AMZN: 0.00%
META: 51.51%

✅ Expected Annual Return: 27.63%
📉 Annual Volatility: 30.34%
🌟 Sharpe Ratio: 0.91
